# Forward Model Hyperparameter Tuning (Wrought only)

Runs systematic hyperparameter search per target: for each property (UTS, Yield, Conductivity, etc.) finds the best model type among XGBoost, Random Forest, and Gradient Boosting (plus **HistGradientBoosting** and **ExtraTrees** for YS and Fatigue) and its best hyperparameters. Saves per-target results to `hyperparams_config.json` under `wrought.by_target`.

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    ExtraTreesRegressor,
)
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
import os

warnings.filterwarnings('ignore')

# Data path - use local or Colab paths
WROUGHT_PATH = 'wrought_alloys_final.csv'

try:
    from utils import load_hyperparams, save_hyperparams, save_per_target_hyperparams
except ImportError:
    import sys
    sys.path.insert(0, os.getcwd())
    from utils import load_hyperparams, save_hyperparams, save_per_target_hyperparams

In [2]:
# Load Wrought Data
INPUT_COLS = ['Al', 'Si', 'Fe', 'Cu', 'Mn', 'Mg', 'Cr', 'Ni', 'Zn', 'Ga', 'V', 'Ti']

def load_wrought(path):
    if path.endswith('.xlsx') or path.endswith('.xls'):
        df = pd.read_excel(path)
    else:
        try:
            with open(path, 'rb') as f:
                head = f.read(2)
            if head == b'PK':
                df = pd.read_excel(path)  # File is xlsx with .csv extension
            else:
                try:
                    df = pd.read_csv(path, encoding='utf-8')
                except UnicodeDecodeError:
                    df = pd.read_csv(path, encoding='latin-1')
        except Exception:
            df = pd.read_excel(path)
    exclude = INPUT_COLS + ['Series', 'Parent Alloy', 'Alloy Name', 'AlloyNumber', 'Temper', 'El (%)']
    targets = [c for c in df.columns if c not in exclude]
    for col in INPUT_COLS + targets:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df[INPUT_COLS] = df[INPUT_COLS].fillna(0.0)
    return df, targets

print('Loading wrought dataset...')
df_wrought, targets_wrought = load_wrought(WROUGHT_PATH) if os.path.exists(WROUGHT_PATH) else (None, [])
if df_wrought is not None:
    print(f'Wrought: {df_wrought.shape}, targets: {len(targets_wrought)}')
else:
    print('Wrought dataset not found')

Loading wrought dataset...


Wrought: (868, 31), targets: 13


In [3]:
# Parameter grids for RandomizedSearchCV (expanded for tree models)
N_ITER_STANDARD = 15
N_ITER_HARD = 28
CV_STANDARD = 3
CV_HARD = 5

def is_hard_target(col_name):
    """YS and Fatigue get extra models, more CV folds, and more search iterations."""
    n = col_name.lower()
    return any(k in n for k in ('ys (mpa)', 'fatigue'))

XGB_PARAMS = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.9],
    'colsample_bytree': [0.7, 0.9],
    'min_child_weight': [1, 3],
    'reg_alpha': [0.01, 0.1, 1],
    'reg_lambda': [0.1, 1, 10],
    'gamma': [0, 0.1, 0.2],
}
RF_PARAMS = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.5],
}
GB_PARAMS = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', None],
}
HGB_PARAMS = {
    'max_iter': [100, 200, 300],
    'max_depth': [3, 5, 7, None],
    'learning_rate': [0.01, 0.05, 0.1],
    'min_samples_leaf': [1, 5, 20],
    'l2_regularization': [0.0, 0.1, 1.0],
    'max_bins': [63, 127, 255],
}
ET_PARAMS = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.5],
}

def tune_and_select_best_per_target(df, targets, input_cols, min_samples=30):
    """
    Per target: RandomizedSearchCV over model families. Hard targets (YS, Fatigue)
    also search HistGradientBoosting + ExtraTrees with more iterations and CV folds.
    Returns by_target dict: { target_name: { 'model': str, 'params': {...} } }
    """
    by_target = {}
    for target in targets:
        df_t = df.dropna(subset=[target])
        if len(df_t) < min_samples:
            continue
        X = df_t[input_cols]
        y = df_t[target]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        hard = is_hard_target(target)
        n_iter = N_ITER_HARD if hard else N_ITER_STANDARD
        cv = CV_HARD if hard else CV_STANDARD

        candidates = [
            ('XGBoost', xgb.XGBRegressor(objective='reg:squarederror', random_state=42), XGB_PARAMS),
            ('RandomForest', RandomForestRegressor(random_state=42), RF_PARAMS),
            ('GradientBoosting', GradientBoostingRegressor(random_state=42), GB_PARAMS),
        ]
        if hard:
            candidates.extend([
                ('HistGradientBoosting', HistGradientBoostingRegressor(random_state=42), HGB_PARAMS),
                ('ExtraTrees', ExtraTreesRegressor(random_state=42), ET_PARAMS),
            ])

        tag = ' [HARD: +HGB/ET, more search]' if hard else ''
        print(f">> TARGET: {target}{tag}")

        best_r2 = -float('inf')
        best_mae = float('inf')
        best_name = None
        best_params = None

        for name, model, params in candidates:
            search = RandomizedSearchCV(
                model, params, n_iter=n_iter, cv=cv,
                verbose=0, random_state=42, n_jobs=-1,
            )
            search.fit(X_train, y_train)
            pred = search.predict(X_test)
            r2 = r2_score(y_test, pred)
            mae = mean_absolute_error(y_test, pred)
            print(f"    {name}: R²={r2:.4f}, MAE={mae:.2f}")
            if r2 > best_r2:
                best_r2 = r2
                best_mae = mae
                best_name = name
                p = {k: v for k, v in search.best_params_.items()}
                p['random_state'] = 42
                best_params = p

        if best_name is not None:
            by_target[target] = {'model': best_name, 'params': best_params}
            crit = f"R²={best_r2:.4f}, MAE={best_mae:.2f}"
            print(f"  WINNER for {target}: {best_name} ({crit})")
    return by_target

In [4]:
# Tune WROUGHT dataset (per-target best model + params)
print('[=== TUNING WROUGHT (per target) ===]')
wrought_by_target = {}
if df_wrought is not None and len(targets_wrought) > 0:
    wrought_by_target = tune_and_select_best_per_target(
        df_wrought, targets_wrought, INPUT_COLS, min_samples=30
    )
else:
    print('  Skipped (no data)')

[=== TUNING WROUGHT (per target) ===]
>> TARGET: UTS (MPa)


    XGBoost: R²=0.8707, MAE=35.93


    RandomForest: R²=0.8735, MAE=35.43


    GradientBoosting: R²=0.8730, MAE=35.67
  WINNER for UTS (MPa): RandomForest (R²=0.8735, MAE=35.43)
>> TARGET: YS (MPa) [HARD: +HGB/ET, more search]


    XGBoost: R²=0.6716, MAE=51.71


...

    RandomForest: R²=0.9852, MAE=0.31


    GradientBoosting: R²=0.9835, MAE=0.57
  WINNER for EC Volume (% IACS): RandomForest (R²=0.9852, MAE=0.31)
>> TARGET: EC Weight (% IACS)


    XGBoost: R²=0.9851, MAE=1.01


    RandomForest: R²=0.9822, MAE=1.33


    GradientBoosting: R²=0.9709, MAE=1.25
  WINNER for EC Weight (% IACS): XGBoost (R²=0.9851, MAE=1.01)


In [5]:
# Save per-target best to config (wrought.by_target)
if wrought_by_target:
    save_per_target_hyperparams('wrought', wrought_by_target)
    print(f'Saved wrought.by_target ({len(wrought_by_target)} targets) to hyperparams_config.json')
else:
    print('No results to save')

Saved wrought.by_target (13 targets) to hyperparams_config.json
